# GP Regression with Uncertain Inputs

## Introduction

Standard GP regression assumes the training and test inputs are known exactly, and only the targets are noisy. This example demonstrates `gpytorch.kernels.UncertainKernel`, which implements the noise-aware RBF kernel from Dallaire, Besse and Chaib draa, "Learning Gaussian Process Models from Uncertain Data" (ICONIP 2009). Each input is encoded as a `(mean, variance)` pair instead of a single value, and the kernel folds that variance into the covariance computation directly. `gpytorch.means.UncertainZeroMean` is the matching mean module that knows to read only the mean column of these paired inputs.

Note: at prediction time, test inputs must also be padded with a variance column (zero if the test input is known exactly), and training/test batches can be different sizes.

In [ ]:
import math

import torch
from matplotlib import pyplot as plt

import gpytorch

%matplotlib inline

## Set up training data

Inputs are encoded as `(mean, variance)` pairs: we'll use 20 points regularly spaced in `[0, 1]`, with input variance shrinking the closer we get to `1`.

In [ ]:
train_x_mean = torch.linspace(0, 1, 20)
train_x_stdv = torch.linspace(0.03, 0.01, 20)
train_x = torch.stack((train_x_mean, train_x_stdv), dim=1)

# True function is sin(2*pi*x) with Gaussian noise on the targets
train_y = torch.sin(train_x_mean * (2 * math.pi)) + torch.randn(train_x_mean.size()) * 0.2

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.errorbar(train_x_mean, train_y, xerr=(train_x_stdv * 2), fmt="k*", label="Train Data")
ax.legend()

## Set up the model

We pair `gpytorch.means.UncertainZeroMean` with `gpytorch.kernels.ScaleKernel(gpytorch.kernels.UncertainKernel())` so both modules know to split the `(mean, variance)` input columns correctly.

In [ ]:
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.UncertainZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.UncertainKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(train_x, train_y, likelihood)

## Train the model

In [ ]:
training_iter = 500

model.train()
likelihood.train()

# Use the Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    print(
        "Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f"
        % (
            i + 1,
            training_iter,
            loss.item(),
            model.covar_module.base_kernel.lengthscale.item(),
            model.likelihood.noise.item(),
        )
    )
    optimizer.step()

## Make predictions

Test inputs must also be padded with a variance column (zero here, since the test locations are known exactly), and must match the training batch size.

In [ ]:
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.fast_pred_var(), gpytorch.settings.lazily_evaluate_kernels(False):
    test_x = torch.linspace(0, 1, 20)
    test_x_uncertain = torch.stack((test_x, torch.zeros_like(test_x)), dim=1)
    observed_pred = likelihood(model(test_x_uncertain))

In [ ]:
with torch.no_grad():
    f, ax = plt.subplots(1, 1, figsize=(8, 3))

    lower, upper = observed_pred.confidence_region()
    ax.errorbar(train_x_mean.numpy(), train_y.numpy(), xerr=train_x_stdv.numpy(), fmt="k*")
    ax.plot(test_x.numpy(), observed_pred.mean.numpy(), "b")
    ax.fill_between(test_x.numpy(), lower.numpy(), upper.numpy(), alpha=0.5)
    ax.set_ylim([-3, 3])
    ax.legend(["Observed Data", "Mean", "Confidence"])

In [ ]:
test_y = torch.sin(test_x * (2 * math.pi))
torch.mean(torch.abs(test_y - observed_pred.mean))